В цьому домашньому завданні ми знову працюємо з даними з нашого змагання ["Bank Customer Churn Prediction (DLU Course)"](https://www.kaggle.com/t/7c080c5d8ec64364a93cf4e8f880b6a0).

Тут ми побудуємо рішення задачі класифікації з використанням алгоритмів бустингу: XGBoost та LightGBM, а також використаємо бібліотеку HyperOpt для оптимізації гіперпараметрів.

0. Зчитайте дані `train.csv` в змінну `raw_df` та скористайтесь наведеним кодом нижче аби розділити дані на трнувальні та валідаційні і розділити дані на ознаки з матириці Х та цільову змінну. Назви змінних `train_inputs, train_targets, train_inputs, train_targets` можна змінити на ті, які Вам зручно.

  Наведений скрипт - частина отриманого мною скрипта для обробки даних. Ми тут не викнуємо масштабування та обробку категоріальних змінних, бо хочемо це делегувати алгоритмам, які будемо використовувати. Якщо щось не розумієте в наведених скриптах, рекомендую розібратись: навичка читати код - важлива складова роботи в машинному навчанні.

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
import pandas as pd
import os

data_dir = '/content/drive/MyDrive/ML1/data/kaggle_data'
print(os.listdir(data_dir))          # спершу дивлюсь

raw_df = pd.read_csv(os.path.join(data_dir, 'train.csv'))

['train.csv', 'test.csv', 'sample_submission.csv', 'submission2.csv', 'processed', 'submission_poly.csv', 'submission_poly4.csv', 'submission_poly4_l1.csv']


In [6]:
print(raw_df.shape)
print(raw_df.columns.tolist())
print(raw_df[['id', 'CustomerId', 'Surname']].nunique())
print(raw_df.isna().sum().sum())

(15000, 14)
['id', 'CustomerId', 'Surname', 'CreditScore', 'Geography', 'Gender', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary', 'Exited']
id            15000
CustomerId     6348
Surname         764
dtype: int64
0


In [7]:
target_col = 'Exited'

drop_cols = ['id', 'CustomerId', 'Surname']
input_cols = [c for c in raw_df.columns if c not in drop_cols + [target_col]]
print(input_cols)

['CreditScore', 'Geography', 'Gender', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary']


In [8]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from typing import Tuple, Dict, Any


def split_train_val(df: pd.DataFrame, target_col: str, test_size: float = 0.2, random_state: int = 42) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Split the dataframe into training and validation sets.

    Args:
        df (pd.DataFrame): The raw dataframe.
        target_col (str): The target column for stratification.
        test_size (float): The proportion of the dataset to include in the validation split.
        random_state (int): Random state for reproducibility.

    Returns:
        Tuple[pd.DataFrame, pd.DataFrame]: Training and validation dataframes.
    """
    train_df, val_df = train_test_split(df, test_size=test_size, random_state=random_state, stratify=df[target_col])
    return train_df, val_df


def separate_inputs_targets(df: pd.DataFrame, input_cols: list, target_col: str) -> Tuple[pd.DataFrame, pd.Series]:
    """
    Separate inputs and targets from the dataframe.

    Args:
        df (pd.DataFrame): The dataframe.
        input_cols (list): List of input columns.
        target_col (str): Target column.

    Returns:
        Tuple[pd.DataFrame, pd.Series]: DataFrame of inputs and Series of targets.
    """
    inputs = df[input_cols].copy()
    targets = df[target_col].copy()
    return inputs, targets

In [9]:
train_df, val_df = split_train_val(raw_df, target_col)
X_train, train_targets = separate_inputs_targets(train_df, input_cols, target_col)
X_val,   val_targets   = separate_inputs_targets(val_df,   input_cols, target_col)

print(X_train.shape, X_val.shape)
print(train_targets.mean(), val_targets.mean())
print(X_train.dtypes)

(12000, 10) (3000, 10)
0.2035 0.20333333333333334
CreditScore        float64
Geography           object
Gender              object
Age                float64
Tenure             float64
Balance            float64
NumOfProducts      float64
HasCrCard          float64
IsActiveMember     float64
EstimatedSalary    float64
dtype: object


1. В тренувальному та валідаційному наборі перетворіть категоріальні ознаки на тип `category`. Можна це зробити двома способами:
 1. `df[col_name].astype('category')`, як було продемонстровано в лекції
 2. використовуючи метод `pd.Categorical(df[col_name])`

In [10]:
from pandas.api.types import CategoricalDtype

cat_cols = X_train.select_dtypes(include='object').columns.tolist()
print('категоріальні колонки:', cat_cols)

# словник рівнів будується з train і застосовується до обох наборів
cat_types = {c: CategoricalDtype(categories=sorted(X_train[c].dropna().unique()))
             for c in cat_cols}

for c in cat_cols:
    X_train[c] = X_train[c].astype(cat_types[c])
    X_val[c]   = X_val[c].astype(cat_types[c])

# перевірка: тепер .cat існує
for c in cat_cols:
    print(c, list(X_train[c].cat.categories), list(X_val[c].cat.categories))

print(X_train.dtypes)

категоріальні колонки: ['Geography', 'Gender']
Geography ['France', 'Germany', 'Spain'] ['France', 'Germany', 'Spain']
Gender ['Female', 'Male'] ['Female', 'Male']
CreditScore         float64
Geography          category
Gender             category
Age                 float64
Tenure              float64
Balance             float64
NumOfProducts       float64
HasCrCard           float64
IsActiveMember      float64
EstimatedSalary     float64
dtype: object


In [11]:
for c in ['Geography', 'Gender']:
    print(c, list(X_train[c].cat.categories), list(X_val[c].cat.categories))

Geography ['France', 'Germany', 'Spain'] ['France', 'Germany', 'Spain']
Gender ['Female', 'Male'] ['Female', 'Male']


Використала CategoricalDtype з рівнями, взятими з тренувального набору, замість astype('category') окремо на кожному наборі. На цих даних результат ідентичний (рівні збігаються), але при колонці з рідкісними значеннями окремий astype на валідації побудував би інший словник рівнів, і коди категорій розійшлися б з тими, на яких навчена модель.

2. Навчіть на отриманих даних модель `XGBoostClassifier`. Параметри алгоритму встановіть на свій розсуд, ми далі будемо їх тюнити. Рекомендую тренувати не дуже складну модель.

  Опис всіх конфігураційних параметрів XGBoostClassifier - тут https://xgboost.readthedocs.io/en/stable/parameter.html#global-config

  **Важливо:** зробіть такі налаштування `XGBoostClassifier` аби він самостійно обробляв незаповнені значення в даних і обробляв категоріальні колонки.

  Можна також, якщо працюєте в Google Colab, увімкнути можливість використання GPU (`Runtime -> Change runtime type -> T4 GPU`) і встановити параметр `device='cuda'` в `XGBoostClassifier` для пришвидшення тренування бустинг моделі.
  
  Після тренування моделі
  1. Виміряйте точність з допомогою AUROC на тренувальному та валідаційному наборах.
  2. Зробіть висновок про отриману модель: вона хороша/погана, чи є high bias/high variance?
  3. Порівняйте якість цієї моделі з тою, що ви отрмали з використанням DecisionTrees раніше. Чи вийшло покращити якість?

In [12]:
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score

xgb_clf = XGBClassifier(
    n_estimators=50,
    max_depth=3,
    learning_rate=0.1,
    enable_categorical=True,     # вимога умови: категоріальні колонки
    tree_method='hist',          # без нього enable_categorical не працює
    missing=np.nan,
    random_state=42,
    n_jobs=-1,
)

xgb_clf.fit(X_train, train_targets)

train_proba = xgb_clf.predict_proba(X_train)[:, 1]
val_proba   = xgb_clf.predict_proba(X_val)[:, 1]

train_auc = roc_auc_score(train_targets, train_proba)
val_auc   = roc_auc_score(val_targets,   val_proba)

print(f'train AUROC: {train_auc:.4f}')
print(f'val AUROC:   {val_auc:.4f}')

train AUROC: 0.9386
val AUROC:   0.9365


Модель XGBoost з 50 дерев глибини 3 дала AUROC 0.9386 на тренуванні і 0.9365 на валідації. Розрив = 0.0021, тобто перенавчання немає, а обидва числа високі не самі по собі, а відносно того, що ці дані взагалі дозволяють і це видно з лідерборду. Тому це не high bias і не high variance, модель збалансована.

Порівняно з моїм найкращим деревом (val 0.9230), приріст склав 0.0135.

Вимогу самостійно обробляти пропуски виконано через enable_categorical=True і прописаний явно дефолтний missing=np.nan

Категоріальні колонки передані типом category з рівнями, взятими з тренувального набору.

3. Використовуючи бібліотеку `Hyperopt` і приклад пошуку гіперпараметрів для `XGBoostClassifier` з лекції знайдіть оптимальні значення гіперпараметрів `XGBoostClassifier` для нашої задачі. Задайте свою сітку гіперпараметрів виходячи з тих параметрів, які ви б хотіли перебрати. Поставте кількість раундів в підборі гіперпараметрів рівну **20**.

  **Увага!** Для того, аби скористатись hyperopt, нам треба задати функцію `objective`. В ній ми маємо задати loss - це може будь-яка метрика, але бажано використовувтаи ту, яка цільова в вашій задачі. Чим менший лосс - тим ліпша модель на думку hyperopt. Тож, тут нам треба задати loss - негативне значення AUROC. В лекції ми натомість використовували Accuracy.

  Після успішного завершення пошуку оптимальних гіперпараметрів
    - виведіть найкращі значення гіперпараметрів
    - створіть в окремій зміній `final_clf` модель `XGBoostClassifier` з найкращими гіперпараметрами
    - навчіть модель `final_clf`
    - оцініть якість моделі `final_clf` на тренувальній і валідаційній вибірках з допомогою AUROC.
    - зробіть висновок про якість моделі. Чи стала вона краще порівняно з попереднім пунктом (2) цього завдання?

In [13]:
import numpy as np
from hyperopt import fmin, tpe, hp, Trials, space_eval, STATUS_OK
from sklearn.model_selection import cross_val_score, StratifiedKFold

space = {
    'max_depth':        hp.quniform('max_depth', 2, 8, 1),
    'n_estimators':     hp.quniform('n_estimators', 50, 1200, 50),
    'learning_rate':    hp.loguniform('learning_rate', np.log(0.01), np.log(0.3)),
    'min_child_weight': hp.quniform('min_child_weight', 1, 20, 1),
    'subsample':        hp.uniform('subsample', 0.6, 1.0),
    'colsample_bytree': hp.uniform('colsample_bytree', 0.6, 1.0),
    'gamma':            hp.uniform('gamma', 0, 5),
    'reg_lambda':       hp.loguniform('reg_lambda', np.log(0.1), np.log(10.0)),
}

INT_PARAMS = ('max_depth', 'n_estimators', 'min_child_weight')
FIXED = dict(enable_categorical=True, tree_method='hist',
             random_state=42, n_jobs=-1)

cv = StratifiedKFold(5, shuffle=True, random_state=42)

def objective(params):
    p = {**params, **{k: int(params[k]) for k in INT_PARAMS}}
    model = XGBClassifier(**FIXED, **p)
    auc = cross_val_score(model, X_train, train_targets,
                          cv=cv, scoring='roc_auc').mean()
    return {'loss': -auc, 'status': STATUS_OK}

trials = Trials()
best = fmin(objective, space, algo=tpe.suggest, max_evals=20,
            trials=trials, rstate=np.random.default_rng(42))

best_params = space_eval(space, best)
best_params = {**best_params, **{k: int(best_params[k]) for k in INT_PARAMS}}
print('Найкращі гіперпараметри:', best_params)
print('Найкращий CV AUROC:', -min(t['result']['loss'] for t in trials.trials))

100%|██████████| 20/20 [00:59<00:00,  3.00s/trial, best loss: -0.9351781437380143]
Найкращі гіперпараметри: {'colsample_bytree': 0.6021952133019314, 'gamma': 4.424409870627135, 'learning_rate': 0.013081811333115652, 'max_depth': 5, 'min_child_weight': 2, 'n_estimators': 1050, 'reg_lambda': 1.646259614479311, 'subsample': 0.7201685419162034}
Найкращий CV AUROC: 0.9351781437380143


In [14]:
scores = cross_val_score(XGBClassifier(**FIXED, **best_params),
                         X_train, train_targets, cv=cv, scoring='roc_auc')
print(scores.round(4), scores.mean().round(4), scores.std().round(4))

[0.9235 0.9324 0.945  0.9402 0.9348] 0.9352 0.0073


In [15]:
final_clf = XGBClassifier(**FIXED, **best_params)
final_clf.fit(X_train, train_targets)

print('train AUROC:', roc_auc_score(train_targets, final_clf.predict_proba(X_train)[:, 1]))
print('val AUROC:  ', roc_auc_score(val_targets,   final_clf.predict_proba(X_val)[:, 1]))

train AUROC: 0.9440056389208931
val AUROC:   0.9373489265381713


Перший запуск пошуку впер n_estimators у верхню межу сітки (400 при діапазоні до 400), тому я розширила діапазон до 1200 і перезапустила. Оптимум перемістився на 1050 дерев, але CV AUROC при цьому зріс із 0.935124 до 0.935178, тобто на п'ять стотисячних. Заодно learning_rate упав із 0.063 до 0.013: це не інший розв'язок, а той самий, просто записаний меншим кроком і більшою кількістю дерев. Цільова функція вздовж цієї пари виявилась плоскою, і пошук блукав по плато, а не знаходив щось краще.

Щоб не покладатись на враження, я виміряла розкид по фолдах для найкращої моделі: std дорівнює 0.0073 при середньому 0.9352, а окремі фолди розійшлися від 0.9235 до 0.9450. Різниця між двома запусками пошуку менша за цей розкид приблизно в сто тридцять разів. Показово, що 0.0073 майже збігається з 0.008, які я раніше отримала зовсім іншим способом, через розбіжність валідації, публічного і приватного лідербордів на цих самих даних.

Фінальна модель після тюнингу дала train 0.9440 і val 0.9373 проти 0.9386 і 0.9365 у простої моделі з пункту 2. Приріст на валідації 0.0008, тобто одна дев'ята від власної похибки вимірювання, а розрив між train і val при цьому потроївся, з 0.0021 до 0.0067. За ціну в двадцять один раз більшої моделі (1050 дерев проти 50) тюнинг не дав нічого, зате почав зміщувати модель у бік перенавчання. Тому я вважаю, що краще залишити просту модель з пункту 2: коли розкид по фолдах перекриває різницю між кандидатами, правильний вибір це простіша модель на плато.

4. Навчіть на наших даних модель LightGBM. Параметри алгоритму встановіть на свій розсуд, ми далі будемо їх тюнити. Рекомендую тренувати не дуже складну модель.

  Опис всіх конфігураційних параметрів LightGBM - тут https://lightgbm.readthedocs.io/en/latest/Parameters.html

  **Важливо:** зробіть такі налаштування LightGBM аби він самостійно обробляв незаповнені значення в даних і обробляв категоріальні колонки.

  Аби передати категоріальні колонки в LightGBM - необхідно виявити їх індекси і передати в параметрі `cat_feature=cat_feature_indexes`

  Після тренування моделі
  1. Виміряйте точність з допомогою AUROC на тренувальному та валідаційному наборах.
  2. Зробіть висновок про отриману модель: вона хороша/погана, чи є high bias/high variance?
  3. Порівняйте якість цієї моделі з тою, що ви отрмали з використанням XGBoostClassifier раніше. Чи вийшло покращити якість?

In [16]:
from lightgbm import LGBMClassifier

lgb_clf = LGBMClassifier(
    n_estimators=50,
    num_leaves=8,          # у LightGBM головний обмежувач складності, не max_depth
    learning_rate=0.1,
    min_child_samples=30,
    random_state=42,
    n_jobs=-1,
    verbose=-1,            # інакше завалить вивід службовими рядками
)

lgb_clf.fit(X_train, train_targets)

# перевірка, що категоріальні справді розпізнані
print(lgb_clf.booster_.params.get('categorical_feature', 'не задано явно'))
print(X_train.dtypes[X_train.dtypes == 'category'].index.tolist())

train_proba = lgb_clf.predict_proba(X_train)[:, 1]
val_proba   = lgb_clf.predict_proba(X_val)[:, 1]
print('train AUROC:', roc_auc_score(train_targets, train_proba))
print('val AUROC:  ', roc_auc_score(val_targets,   val_proba))

не задано явно
['Geography', 'Gender']
train AUROC: 0.9396685891506984
val AUROC:   0.9365326839975308


In [17]:
pd.Series(lgb_clf.feature_importances_, index=X_train.columns).sort_values(ascending=False)

,0
NumOfProducts,110
Age,90
IsActiveMember,38
Balance,38
CreditScore,24
Geography,23
Gender,21
Tenure,4
EstimatedSalary,2
HasCrCard,0


> LightGBM з 50 дерев і 8 листками дав train AUROC 0.9397 і val 0.9365. Порівняно з XGBoost із пункту 2 (0.9386 і 0.9365) різниця на валідації склала три стотисячних, тобто моделі показали однакову якість: розкид по фолдах у мене дорівнює 0.0073, а це у двісті разів більше за різницю. Покращити якість не вийшло, і жодна з двох бібліотек тут не має переваги.

> Розрив між train і val 0.0031, обидва числа близькі до стелі сигналу в цих даних (верхівка лідерборду), тому висновок той самий, що й раніше: ні high bias, ні high variance, модель збалансована.

> Категоріальні колонки LightGBM обробив нативно, без параметра `cat_feature` з умови (такого параметра в бібліотеці немає, правильна назва `categorical_feature`). Оскільки `Geography` і `Gender` уже мають тип `category`, LightGBM розпізнає їх сам, що видно з важливостей ознак: 23 і 21 розбиття відповідно.

5. Використовуючи бібліотеку `Hyperopt` і приклад пошуку гіперпараметрів для `LightGBM` з лекції знайдіть оптимальні значення гіперпараметрів `LightGBM` для нашої задачі. Задайте свою сітку гіперпараметрів виходячи з тих параметрів, які ви б хотіли перебрати. Поставте кількість раундів в підборі гіперпараметрів рівну **10**.

  **Увага!** Для того, аби скористатись hyperopt, нам треба задати функцію `objective`. І тут ми також ставимо loss - негативне значення AUROC, як і при пошуці гіперпараметрів для XGBoost. До речі, можна спробувати написати код так, аби в objective передавати лише модель і не писати схожий код двічі :)

  Після успішного завершення пошуку оптимальних гіперпараметрів
    - виведіть найкращі значення гіперпараметрів
    - створіть в окремій зміній `final_lgb_clf` модель `LightGBM` з найкращими гіперпараметрами
    - навчіть модель `final_lgb_clf`
    - оцініть якість моделі `final_lgb_clf` на тренувальній і валідаційній вибірках з допомогою AUROC.
    - зробіть висновок про якість моделі. Чи стала вона краще порівняно з попереднім пунктом (4) цього завдання?

In [19]:
from lightgbm import LGBMClassifier

space_lgb = {
    'num_leaves':        hp.quniform('num_leaves', 8, 128, 4),
    'learning_rate':     hp.loguniform('learning_rate', np.log(0.01), np.log(0.3)),
    'min_child_samples': hp.quniform('min_child_samples', 5, 400, 5),
    'subsample':         hp.uniform('subsample', 0.6, 1.0),
    'colsample_bytree':  hp.uniform('colsample_bytree', 0.6, 1.0),
    'reg_lambda':        hp.loguniform('reg_lambda', np.log(0.01), np.log(10.0)),
    'min_split_gain':    hp.uniform('min_split_gain', 0, 1),
}

INT_LGB = ('num_leaves', 'min_child_samples')
FIXED_LGB = dict(n_estimators=300, subsample_freq=1,   # без freq бегінг вимкнений
                 random_state=42, n_jobs=-1, verbose=-1)

def make_objective(model_cls, fixed, int_params):
    def objective(params):
        p = {**params, **{k: int(params[k]) for k in int_params}}
        model = model_cls(**fixed, **p)
        auc = cross_val_score(model, X_train, train_targets,
                              cv=cv, scoring='roc_auc').mean()
        return {'loss': -auc, 'status': STATUS_OK}
    return objective

trials_lgb = Trials()
best_lgb_raw = fmin(make_objective(LGBMClassifier, FIXED_LGB, INT_LGB),
                    space_lgb, algo=tpe.suggest, max_evals=10,
                    trials=trials_lgb, rstate=np.random.default_rng(42))

best_lgb = space_eval(space_lgb, best_lgb_raw)
best_lgb = {**best_lgb, **{k: int(best_lgb[k]) for k in INT_LGB}}
print('Найкращі гіперпараметри:', best_lgb)
print('Найкращий CV AUROC:', -min(t['result']['loss'] for t in trials_lgb.trials))

100%|██████████| 10/10 [00:39<00:00,  3.96s/trial, best loss: -0.9318798470368386]
Найкращі гіперпараметри: {'colsample_bytree': 0.7845735941946608, 'learning_rate': 0.06788033425401534, 'min_child_samples': 365, 'min_split_gain': 0.7231479244442287, 'num_leaves': 60, 'reg_lambda': 0.030413914859824455, 'subsample': 0.9018909441059156}
Найкращий CV AUROC: 0.9318798470368386


In [20]:
# CV для своєї простої моделі з пункту 4
scores_simple = cross_val_score(lgb_clf, X_train, train_targets, cv=cv, scoring='roc_auc')
print(scores_simple.round(4), scores_simple.mean().round(5), scores_simple.std().round(4))

[0.9231 0.933  0.9444 0.9383 0.9339] 0.93454 0.007


In [21]:
final_lgb_clf = LGBMClassifier(**FIXED_LGB, **best_lgb)
final_lgb_clf.fit(X_train, train_targets)

print('train AUROC:', roc_auc_score(train_targets, final_lgb_clf.predict_proba(X_train)[:, 1]))
print('val AUROC:  ', roc_auc_score(val_targets,   final_lgb_clf.predict_proba(X_val)[:, 1]))

train AUROC: 0.953759293448559
val AUROC:   0.934388504012621


Сітку для LightGBM я будувала не з нуля, а з урахуванням двох спостережень із попередніх пунктів. По-перше, n_estimators зафіксувала на 300 і не тюнила: у пункті 3 було видно, що ця величина утворює плато з learning_rate (крок упав вп'ятеро, дерев стало в 2.5 раза більше, результат не змінився), тому перебирати обидві означає витрачати вимірювання на блукання вздовж однієї лінії. По-друге, num_leaves розширила щедро, бо сума важливостей ручної моделі дорівнювала рівно 350, тобто 50 дерев по 7 розбиттів, і всі дерева вичерпали бюджет із 8 листків.

Перший запуск уперся в межу: min_child_samples дорівнював 100 при верхній межі 100, до того ж напрямок був у бік більшої регуляризації. Я розширила діапазон до 400 і перезапустила, і результат став гіршим, 0.93188 проти 0.93355. Пояснення просте: зміна простору означає інші десять випадкових точок, а не ті самі плюс нові. При 7 параметрах і 10 спробах навіть по 3 значення на вісь дали б 3^7 = 2187 комбінацій, тобто я дивлюсь менше 0,5% простору, а TPE до двадцятої спроби взагалі не вмикає свою модель. Пункт 5 за таких умов це випадковий пошук із 10 точок, і різниця 0.0017 між двома запусками при розкиді по фолдах 0.0070 нічого не означає.

Чесне порівняння на однакових фолдах виглядає так: ручна LightGBM дає CV 0.93454, тюнена 0.93188, тюнений XGBoost 0.93518. Тобто зібрана руками модель виявилась кращою за обидва результати пошуку.

Фінальна модель final_lgb_clf дала train 0.9538 і val 0.9344. Це найвищий train і найнижчий val із чотирьох моделей, а розрив 0.0194 ушестеро більший, ніж у моєї ручної LightGBM (0.0031). Тут варто розрізняти статус двох чисел: розбіжність по val між моделями тоне в шумі, а розрив між train і val міряється на одній і тій самій моделі, тому він надійний. Отже пошук видав модель із 300 дерев і 60 листками, яка стала краще запам'ятовувати тренування і гірше узагальнювати. Якість не покращилась, а в цьому конкретному випадку погіршилась, і причина не в бібліотеці: прості моделі вже стоять біля стелі сигналу цих даних (по верхівці лідерборду), тому рухатись просто нема куди.

6. Оберіть модель з експериментів в цьому ДЗ і зробіть новий `submission` на Kaggle та додайте код для цього і скріншот скора на публічному лідерборді.
  
  **Напишіть коментар, чому ви обрали саме цю модель?**

  І я вас вітаю - це останнє завдання з цим набором даних 💪 На цьому етапі корисно проаналізувати, які моделі показали себе найкраще і подумати, чому.

In [23]:
test_df = pd.read_csv(os.path.join(data_dir, 'test.csv'))
sample   = pd.read_csv(os.path.join(data_dir, 'sample_submission.csv'))
chosen_model = xgb_clf          # проста XGBoost з пункту 2

X_test = test_df[input_cols].copy()
for c in cat_cols:
    X_test[c] = X_test[c].astype(cat_types[c])     # ті самі рівні, що в train

preds = pd.DataFrame({
    'id': test_df['id'],
    'Exited': chosen_model.predict_proba(X_test)[:, 1]
})

submission = sample.copy()
submission['Exited'] = submission['id'].map(preds.set_index('id')['Exited'])

print(submission['Exited'].isna().sum())      # має бути 0
print(submission['Exited'].mean())            # має бути близько 0.204
print(submission['Exited'].nunique())         # має бути багато, не 2
submission.to_csv('submission_xgb.csv', index=False)
submission.head()

0
0.20938408
1938


,id,Exited
0,15000,0.099022
1,15001,0.025704
2,15002,0.070138
3,15003,0.519077
4,15004,0.049040


Для submit я обрала просту модель XGBoost із пункту 2: 50 дерев, глибина 3, без тюнингу.

Причина в тому, що всі чотири моделі цього ДЗ лежать на одному плато. На однакових фолдах вони дали CV від 0.93188 до 0.93518, тобто розмах 0.0033, тоді як стандартне відхилення по фолдах однієї моделі дорівнює 0.0070. Різниця між найкращою і найгіршою моделлю вдвічі менша за похибку вимірювання будь-якої з них, отже вибирати за метрикою тут нема з чого: я б обирала найвдаліший жереб, а не найкращу модель.

Коли різниці немає, я беру найпростішу модель на плато. У простої XGBoost найменший розрив між train і val (0.0021 проти 0.0067 у тюненої XGBoost і 0.0194 у тюненої LightGBM), вона у двадцять один раз менша за тюнену і навчається за секунди. Додатковий аргумент з мого попереднього досвіду на цих даних: порядок моделей у мене вже одного разу перевернувся між публічним і приватним лідербордом, а просідає першою саме складна модель.

Загальний висновок по всьому набору: прості моделі вже стоять біля стелі сигналу цих даних, тому тюнинг не мав звідки взяти приріст. Найбільше на цих даних дало не налаштування гіперпараметрів, а перехід від одного дерева до ансамблю: 0.9230 проти 0.9365.

In [24]:
from google.colab import files
files.download('submission_xgb.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>